# PDF Parsing Service (Jupyter)

Ноутбук реализует сервис парсинга PDF с поддержкой:
- извлечения текста (PyMuPDF + fallback на pdfplumber),
- очистки текста,
- извлечения структурированных полей через regex,
- извлечения таблиц и нормализации в `pandas.DataFrame`,
- пакетной обработки PDF,
- валидации качества и экспорта результатов.

## 1) Установка и импорт зависимостей

In [1]:
%pip install -q pymupdf pdfplumber pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [1]:
import json
import logging
import re
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List, Optional

import fitz  # PyMuPDF
import pandas as pd
import pdfplumber

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s'
)
logger = logging.getLogger('pdf-parser-service')
logger.info('Зависимости импортированы, логирование настроено.')

2026-04-04 23:22:58,322 | INFO | pdf-parser-service | Зависимости импортированы, логирование настроено.


## 2) Конфигурация путей и параметров сервиса

In [2]:
@dataclass
class ParserConfig:
    input_dir: Path = Path('./input_pdfs')
    output_dir: Path = Path('./output')
    file_pattern: str = '*.pdf'
    encoding: str = 'utf-8'
    parse_tables: bool = True
    min_text_length: int = 80
    min_quality_score: float = 0.55
    max_return_text_chars: int = 20_000
    overwrite: bool = False


config = ParserConfig()
config.input_dir.mkdir(parents=True, exist_ok=True)
config.output_dir.mkdir(parents=True, exist_ok=True)

logger.info('ParserConfig инициализирован: %s', config)

2026-04-04 23:23:09,347 | INFO | pdf-parser-service | ParserConfig инициализирован: ParserConfig(input_dir=PosixPath('input_pdfs'), output_dir=PosixPath('output'), file_pattern='*.pdf', encoding='utf-8', parse_tables=True, min_text_length=80, min_quality_score=0.55, max_return_text_chars=20000, overwrite=False)


## 3) Извлечение текста из PDF (PyMuPDF / pdfplumber)

In [3]:
def extract_text_pymupdf(pdf_path: Path) -> Dict[str, Any]:
    pages: List[Dict[str, Any]] = []
    metadata: Dict[str, Any] = {}

    with fitz.open(pdf_path) as doc:
        metadata = doc.metadata or {}
        for i, page in enumerate(doc, start=1):
            txt = page.get_text('text') or ''
            pages.append({'page': i, 'text': txt})

    full_text = '\n'.join(p['text'] for p in pages).strip()
    return {
        'text': full_text,
        'pages': pages,
        'metadata': metadata,
        'engine': 'pymupdf'
    }


def extract_text_pdfplumber(pdf_path: Path) -> Dict[str, Any]:
    pages: List[Dict[str, Any]] = []
    metadata: Dict[str, Any] = {}

    with pdfplumber.open(pdf_path) as doc:
        metadata = dict(doc.metadata or {})
        for i, page in enumerate(doc.pages, start=1):
            txt = page.extract_text() or ''
            pages.append({'page': i, 'text': txt})

    full_text = '\n'.join(p['text'] for p in pages).strip()
    return {
        'text': full_text,
        'pages': pages,
        'metadata': metadata,
        'engine': 'pdfplumber'
    }


def extract_text_from_pdf(pdf_path: Path) -> Dict[str, Any]:
    errors: List[str] = []

    try:
        result = extract_text_pymupdf(pdf_path)
        if result['text'].strip():
            result['errors'] = errors
            return result
        errors.append('pymupdf_extracted_empty_text')
    except Exception as e:
        errors.append(f'pymupdf_error: {e}')

    try:
        result = extract_text_pdfplumber(pdf_path)
        result['errors'] = errors
        return result
    except Exception as e:
        errors.append(f'pdfplumber_error: {e}')

    return {
        'text': '',
        'pages': [],
        'metadata': {},
        'engine': 'none',
        'errors': errors,
    }

## 4) Предобработка и очистка извлечённого текста

In [4]:
def clean_text(text: str) -> str:
    text = text.replace('\x00', ' ')
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{2,}', '\n', text)
    text = re.sub(r'[•·▪▫]', '-', text)
    return text.strip()


def normalize_dates(text: str) -> str:
    # dd.mm.yyyy -> yyyy-mm-dd
    def _to_iso(match: re.Match) -> str:
        value = match.group(0)
        for fmt in ('%d.%m.%Y', '%d-%m-%Y', '%d/%m/%Y'):
            try:
                return datetime.strptime(value, fmt).strftime('%Y-%m-%d')
            except ValueError:
                continue
        return value

    return re.sub(r'\b\d{2}[./-]\d{2}[./-]\d{4}\b', _to_iso, text)


def normalize_numbers(text: str) -> str:
    # 1 234 567,89 -> 1234567.89
    text = re.sub(r'(?<=\d)\s+(?=\d{3}(\D|$))', '', text)
    text = re.sub(r'(\d+),(\d{2})\b', r'\1.\2', text)
    return text


def preprocess_text(text: str) -> str:
    text = clean_text(text)
    text = normalize_dates(text)
    text = normalize_numbers(text)
    return text

## 5) Парсинг только IELTS score через регулярные выражения

In [11]:
IELTS_PATTERNS: List[str] = [
    # IELTS Overall Band Score: 7.5
    r"overall\s*band\s*score\D{0,25}([0-9](?:\s*[\.,]\s*[0-9])?)",
    # Overall: 8.0
    r"overall\D{0,20}([0-9](?:\s*[\.,]\s*[0-9])?)",
    # IELTS ... Overall ... 6.5
    r"ielts\D{0,40}overall\D{0,20}([0-9](?:\s*[\.,]\s*[0-9])?)",
]


def _normalize_score(raw: str) -> Optional[float]:
    value = re.sub(r"\s+", "", raw).replace(",", ".")
    try:
        score = float(value)
    except ValueError:
        return None

    # IELTS диапазон 0.0..9.0, обычно шаг 0.5
    if not (0.0 <= score <= 9.0):
        return None
    if abs(score * 2 - round(score * 2)) > 1e-9:
        return None
    return score


def _extract_score_candidates_with_pos(text: str) -> List[tuple[int, float]]:
    candidates: List[tuple[int, float]] = []
    for m in re.finditer(r"\b([0-9](?:[\.,][0-9])?)\b", text):
        score = _normalize_score(m.group(1))
        if score is not None:
            candidates.append((m.start(), score))
    return candidates


def _extract_score_by_overall_position(text: str) -> Optional[float]:
    text_l = text.lower()
    kw = re.search(r"overall\s*band\s*score|overall", text_l)
    if not kw:
        return None

    candidates = _extract_score_candidates_with_pos(text)
    for pos, score in candidates:
        if pos > kw.start():
            return score
    return None


def extract_ielts_from_tables(pdf_path: Path) -> Optional[float]:
    try:
        with pdfplumber.open(pdf_path) as doc:
            for page in doc.pages:
                tables = page.extract_tables() or []
                for table in tables:
                    for row in table:
                        if not row:
                            continue
                        row_cells = [str(c or "") for c in row]
                        row_text = " | ".join(row_cells)
                        row_text_l = row_text.lower()

                        # Ищем строку, где есть Overall/Band/Score
                        if "overall" in row_text_l and ("band" in row_text_l or "score" in row_text_l):
                            for cell in row_cells:
                                m = re.search(r"\b([0-9](?:[\.,][0-9])?)\b", cell)
                                if m:
                                    score = _normalize_score(m.group(1))
                                    if score is not None:
                                        return score
    except Exception as e:
        logger.warning("Табличный парсинг IELTS не удался: %s", e)

    return None


def extract_ielts_score(text: str, pdf_path: Optional[Path] = None) -> Dict[str, Any]:
    text_l = text.lower()

    # 1) Поиск по текстовым шаблонам
    for pattern in IELTS_PATTERNS:
        m = re.search(pattern, text_l, flags=re.IGNORECASE)
        if m:
            score = _normalize_score(m.group(1))
            if score is not None:
                return {
                    'ielts_score': score,
                    'status': 'ok',
                    'error': None,
                    'source': 'text_pattern',
                }

    # 2) Эвристика: взять ближайший score после слова Overall
    positional_score = _extract_score_by_overall_position(text)
    if positional_score is not None:
        return {
            'ielts_score': positional_score,
            'status': 'ok',
            'error': None,
            'source': 'text_positional',
        }

    # 3) Fallback: поиск в таблицах
    if pdf_path is not None:
        table_score = extract_ielts_from_tables(pdf_path)
        if table_score is not None:
            return {
                'ielts_score': table_score,
                'status': 'ok',
                'error': None,
                'source': 'table',
            }

    return {
        'ielts_score': None,
        'status': 'not_found',
        'error': 'IELTS score не найден в тексте/таблицах PDF',
        'source': None,
    }


def parse_pdf_for_ielts(pdf_path: Path, cfg: ParserConfig = config) -> Dict[str, Any]:
    extraction = extract_text_from_pdf(pdf_path)
    cleaned_text = preprocess_text(extraction.get('text', ''))

    if not cleaned_text or len(cleaned_text) < cfg.min_text_length:
        # даже при коротком тексте пробуем таблицы
        ielts = extract_ielts_score(cleaned_text, pdf_path=pdf_path)
        if ielts['ielts_score'] is not None:
            return {
                'file': str(pdf_path),
                'ielts_score': ielts['ielts_score'],
                'status': ielts['status'],
                'error': ielts['error'],
                'source': ielts.get('source'),
                'engine': extraction.get('engine'),
                'pages_count': len(extraction.get('pages', [])),
            }

        return {
            'file': str(pdf_path),
            'ielts_score': None,
            'status': 'bad_quality',
            'error': f'Недостаточно текста для анализа (len={len(cleaned_text)})',
            'source': None,
            'engine': extraction.get('engine'),
            'pages_count': len(extraction.get('pages', [])),
        }

    ielts = extract_ielts_score(cleaned_text, pdf_path=pdf_path)
    return {
        'file': str(pdf_path),
        'ielts_score': ielts['ielts_score'],
        'status': ielts['status'],
        'error': ielts['error'],
        'source': ielts.get('source'),
        'engine': extraction.get('engine'),
        'pages_count': len(extraction.get('pages', [])),
    }

In [20]:
ENT_PATTERNS: List[str] = [
    # ЕНТ: 112
    r"(?:ент|ен\s*т|unt)\s*[:\-]?\s*([0-9]{2,3})",
    # Итоговый балл: 105
    r"(?:итог(?:овый)?\s*балл|общий\s*балл|total\s*score)\s*[:\-]?\s*([0-9]{2,3})",
    # 42 Барлығы/Итого
    r"\b([0-9]{1,3})\b\s*(?:барлығы|итого|всего|total)\b",
]


def _normalize_ent_score(raw: str) -> Optional[int]:
    digits = re.sub(r"\D+", "", raw)
    if not digits:
        return None

    score = int(digits)
    # Стандартный диапазон ЕНТ
    if 0 <= score <= 140:
        return score
    return None


def _extract_ent_by_keyword_position(text: str) -> Optional[int]:
    text_l = text.lower()

    keyword_match = re.search(r"барлығы|итого|всего|total", text_l)
    if not keyword_match:
        return None

    keyword_pos = keyword_match.start()
    candidates: List[tuple[int, int]] = []
    for m in re.finditer(r"\b([0-9]{1,3})\b", text):
        score = _normalize_ent_score(m.group(1))
        if score is None:
            continue
        candidates.append((m.start(), score))

    # Сначала пробуем ближайшее число ПЕРЕД ключевым словом (обычно это итог)
    prev_candidates = [(pos, s) for pos, s in candidates if pos < keyword_pos and keyword_pos - pos <= 120]
    if prev_candidates:
        prev_candidates.sort(key=lambda x: keyword_pos - x[0])
        return prev_candidates[0][1]

    # Если перед ключом ничего нет, берем ближайшее после,
    # но исключаем "из возможных ..." (обычно это максимум, не результат).
    next_candidates = [(pos, s) for pos, s in candidates if pos > keyword_pos and pos - keyword_pos <= 120]
    for pos, score in sorted(next_candidates, key=lambda x: x[0] - keyword_pos):
        local_window = text_l[max(0, pos - 40): pos + 40]
        if re.search(r"мүмкін|возможн|possible|max", local_window):
            continue
        return score

    return None


def extract_ent_score(text_or_pdf: Any) -> Dict[str, Any]:
    # Поддержка обоих вариантов:
    # 1) передан уже извлечённый текст (str)
    # 2) передан путь к PDF (Path/str)
    if isinstance(text_or_pdf, Path):
        extraction = extract_text_from_pdf(text_or_pdf)
        text = preprocess_text(extraction.get('text', ''))
    elif isinstance(text_or_pdf, str):
        path_candidate = Path(text_or_pdf)
        if text_or_pdf.lower().endswith('.pdf') and path_candidate.exists():
            extraction = extract_text_from_pdf(path_candidate)
            text = preprocess_text(extraction.get('text', ''))
        else:
            text = text_or_pdf
    else:
        return {
            'ent_score': None,
            'status': 'error',
            'error': f'Неподдерживаемый тип входа: {type(text_or_pdf).__name__}',
        }

    text_l = text.lower()

    # 1) Прямые regex-паттерны
    for pattern in ENT_PATTERNS:
        m = re.search(pattern, text_l, flags=re.IGNORECASE)
        if m:
            score = _normalize_ent_score(m.group(1))
            if score is not None:
                return {
                    'ent_score': score,
                    'status': 'ok',
                    'error': None,
                }

    # 2) Позиционная эвристика вокруг Барлығы/Итого
    positional_score = _extract_ent_by_keyword_position(text)
    if positional_score is not None:
        return {
            'ent_score': positional_score,
            'status': 'ok',
            'error': None,
        }

    return {
        'ent_score': None,
        'status': 'not_found',
        'error': 'ЕНТ балл не найден в тексте PDF',
    }


def parse_pdf_for_ent(pdf_path: Path, cfg: ParserConfig = config) -> Dict[str, Any]:
    extraction = extract_text_from_pdf(pdf_path)
    cleaned_text = preprocess_text(extraction.get('text', ''))

    if not cleaned_text or len(cleaned_text) < cfg.min_text_length:
        return {
            'file': str(pdf_path),
            'ent_score': None,
            'status': 'bad_quality',
            'error': f'Недостаточно текста для анализа (len={len(cleaned_text)})',
            'engine': extraction.get('engine'),
            'pages_count': len(extraction.get('pages', [])),
        }

    ent = extract_ent_score(cleaned_text)
    return {
        'file': str(pdf_path),
        'ent_score': ent['ent_score'],
        'status': ent['status'],
        'error': ent['error'],
        'engine': extraction.get('engine'),
        'pages_count': len(extraction.get('pages', [])),
    }

## 6) Пример запуска: вытащить только IELTS из одного PDF

In [22]:
# Положи нужный PDF в ./input_pdfs и укажи имя файла
sample_pdf = config.input_dir / '1-2025-32-003904212-2-1.pdf'

if sample_pdf.exists():
    result = extract_ent_score(sample_pdf)
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    print(f'Файл не найден: {sample_pdf}')

{
  "ent_score": 42,
  "status": "ok",
  "error": null
}


In [17]:
# Диагностика: посмотреть извлечённый текст
ex = extract_text_from_pdf(sample_pdf)
preview = preprocess_text(ex.get('text', ''))
print('ENGINE:', ex.get('engine'))
print('TEXT_LEN:', len(preview))
print('\n--- PREVIEW START ---\n')
print(preview[:2500])
print('\n--- PREVIEW END ---')

ENGINE: pymupdf
TEXT_LEN: 823

--- PREVIEW START ---

ТЕСТІЛЕНУШІНІҢ ҰБТ НӘТИЖЕСІ
РЕЗУЛЬТАТЫ ЕНТ ТЕСТИРУЕМОГО
2025-06-07
ҰБТ тапсыру тілі:
Язык сдачи ЕНТ:
орысша
русский
№
Пәндер атауы/Наименование предметов
Жинаған балдары/Набранные баллы
ҰБТ тапсырған мерзімі:
Дата сдачи ЕНТ:
ДУЙСЕН АЛИ БЕГЛАНҰЛЫ
Т.А.Ә. бар болған жағдайда/Ф.И.О. при его наличии
003904212
ТЖК/ИКТ:
051126501305
ЖСН/ИИН:
2025-06-07, 10:53:12
Жасалған күні:
Дата создания:
1-2025-32-003904212-2-1
Бірегей нөмір:
Уникальный номер:
Математика
1
Математика
7
Алгоритмдеу және бағдарламалау негіздері
2
Основы алгоритмизации и программирования
35
ҰБТ нәтижесінің жарамдылық мерзімі
Срок действия результатов ЕНТ до
42
Барлығы/Итого
мүмкін болған
из возможных
балдан
баллов
70
2025-12-31
 дейiн
Тұлғаны куәландыратын құжат болған жағдайда күші бар
Действителен при предъявлении документа, удостоверяющего личность

--- PREVIEW END ---
